In [ ]:
# ==================== 第一部分：环境准备与数据加载 ====================

import os
import time
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, random_split
from torchvision import transforms, datasets, models
from torchvision.models import ResNet18_Weights

# 评估指标库
from sklearn.metrics import (accuracy_score, precision_recall_fscore_support,
                             confusion_matrix, classification_report)

# 固定随机种子
def set_seed(seed=42):
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(42)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# 数据集根目录：优先使用本地数据集，本地不存在时才联网下载
DATA_ROOT = './data'

def cifar10_need_download(root=DATA_ROOT):
    """torchvision 解压后会生成 cifar-10-batches-py 目录；存在即视为本地已就绪，无需联网。"""
    ready = os.path.isdir(os.path.join(root, 'cifar-10-batches-py'))
    if ready:
        print(f"✓ 检测到本地数据集，直接加载: {os.path.abspath(root)}")
    else:
        print(f"⬇ 未检测到本地数据集，将下载到: {os.path.abspath(root)}")
    return not ready

# -------------------- 数据预处理 --------------------
# 训练集：上采样至 224 + 数据增强
train_transform = transforms.Compose([
    transforms.Resize(224),
    transforms.RandomHorizontalFlip(),
    transforms.RandomCrop(224, padding=4),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# 验证集/测试集：仅 Resize + Normalize
test_transform = transforms.Compose([
    transforms.Resize(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# 加载 CIFAR-10 数据集（本地优先；本地已存在则不会联网）
_need_download = cifar10_need_download(DATA_ROOT)
full_train_dataset = datasets.CIFAR10(
    root=DATA_ROOT, train=True, download=_need_download, transform=train_transform
)
test_dataset = datasets.CIFAR10(
    root=DATA_ROOT, train=False, download=_need_download, transform=test_transform
)

# 划分训练集 / 验证集（9:1）
train_size = int(0.9 * len(full_train_dataset))
val_size = len(full_train_dataset) - train_size
train_dataset, val_dataset = random_split(full_train_dataset, [train_size, val_size])

# 验证集的 transform 需要单独设置（random_split 返回 Subset）
val_dataset.dataset.transform = test_transform

# 创建 DataLoader
BATCH_SIZE = 128
NUM_WORKERS = 2

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=NUM_WORKERS, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False,
                        num_workers=NUM_WORKERS, pin_memory=True)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False,
                         num_workers=NUM_WORKERS, pin_memory=True)

# 类别名称
CLASS_NAMES = ['airplane', 'automobile', 'bird', 'cat', 'deer',
               'dog', 'frog', 'horse', 'ship', 'truck']
NUM_CLASSES = len(CLASS_NAMES)

print(f"训练集样本数: {len(train_dataset)}")
print(f"验证集样本数: {len(val_dataset)}")
print(f"测试集样本数: {len(test_dataset)}")
print(f"类别总数: {NUM_CLASSES}")

In [ ]:
# ==================== 第二部分：数据集样本可视化 ====================

# 使用未归一化的原始图像显示（避免颜色异常）
raw_transform = transforms.Compose([
    transforms.Resize(224),
    transforms.ToTensor()
])
# 本地优先：复用第一部分的 DATA_ROOT，本地已存在则不会联网
raw_dataset = datasets.CIFAR10(root=DATA_ROOT, train=True,
                               download=cifar10_need_download(DATA_ROOT),
                               transform=raw_transform)

# 获取每个类别的第一张图
class_to_img = {}
for img, label in raw_dataset:
    if label not in class_to_img:
        class_to_img[label] = img
    if len(class_to_img) == NUM_CLASSES:
        break

# 绘制 2x5 网格
fig, axes = plt.subplots(2, 5, figsize=(15, 6))
axes = axes.flatten()
for i in range(NUM_CLASSES):
    img_tensor = class_to_img[i]
    img_np = img_tensor.permute(1, 2, 0).numpy()
    axes[i].imshow(img_np)
    axes[i].set_title(CLASS_NAMES[i], fontsize=12)
    axes[i].axis('off')
plt.suptitle('CIFAR-10 各类别样本展示 (Resized to 224×224)', fontsize=16)
plt.tight_layout()
plt.show()

In [ ]:
# ==================== 第三部分：ResNet-18 模型构建 ====================

def create_resnet18(num_classes=10, pretrained=True):
    if pretrained:
        model = models.resnet18(weights=ResNet18_Weights.IMAGENET1K_V1)
    else:
        model = models.resnet18(weights=None)
    in_features = model.fc.in_features
    model.fc = nn.Linear(in_features, num_classes)
    return model

model = create_resnet18(num_classes=NUM_CLASSES, pretrained=True).to(device)

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"总参数量: {total_params:,}")
print(f"可训练参数量: {trainable_params:,}")

In [ ]:
# ==================== 第四部分：早停机制与训练辅助函数 ====================

class EarlyStopping:
    """早停机制：当验证指标在 patience 轮内未提升时停止训练"""
    def __init__(self, patience=5, delta=0.001, mode='max', verbose=True):
        """
        Args:
            patience (int): 容忍多少个 epoch 无提升
            delta (float): 提升的最小阈值
            mode (str): 'max'（监控指标越大越好，如准确率）或 'min'（监控指标越小越好，如损失）
            verbose (bool): 是否打印早停信息
        """
        self.patience = patience
        self.delta = delta
        self.mode = mode
        self.verbose = verbose
        self.counter = 0
        self.best_score = None
        self.early_stop = False
        if mode == 'max':
            self.best_score = -float('inf')
        else:
            self.best_score = float('inf')

    def __call__(self, current_score):
        if self.mode == 'max':
            score_improved = current_score > self.best_score + self.delta
        else:
            score_improved = current_score < self.best_score - self.delta

        if score_improved:
            self.best_score = current_score
            self.counter = 0
        else:
            self.counter += 1
            if self.verbose:
                print(f"早停计数器: {self.counter}/{self.patience} (当前最佳: {self.best_score:.4f})")
            if self.counter >= self.patience:
                self.early_stop = True

def train_epoch(model, dataloader, criterion, optimizer, device):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0
    pbar = tqdm(dataloader, desc="Training", leave=False)
    for images, labels in pbar:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * images.size(0)
        _, predicted = torch.max(outputs, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()
        pbar.set_postfix({'loss': loss.item(), 'acc': correct/total})
    epoch_loss = running_loss / total
    epoch_acc = correct / total
    return epoch_loss, epoch_acc

def evaluate(model, dataloader, criterion, device):
    model.eval()
    running_loss = 0.0
    correct = 0
    total = 0
    all_preds = []
    all_labels = []
    with torch.no_grad():
        pbar = tqdm(dataloader, desc="Evaluating", leave=False)
        for images, labels in pbar:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)
            running_loss += loss.item() * images.size(0)
            _, predicted = torch.max(outputs, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
            all_preds.extend(predicted.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
            pbar.set_postfix({'loss': loss.item(), 'acc': correct/total})
    epoch_loss = running_loss / total
    epoch_acc = correct / total
    return epoch_loss, epoch_acc, all_preds, all_labels

In [ ]:
# ==================== 第五部分：模型训练（集成早停） ====================

NUM_EPOCHS = 30               # 设置一个较大的值，早停会自动截断
LEARNING_RATE = 0.001
WEIGHT_DECAY = 1e-4
PATIENCE = 5                  # 连续5个epoch验证准确率不提升则停止

criterion = nn.CrossEntropyLoss()

# CPU 上完整训练极慢；若已有训练好的权重，设 SKIP_TRAINING=True 可直接加载并跳过训练。
SKIP_TRAINING = True
CKPT_PATH = '/kaggle/input/notebooks/liangliguo/cifar/resnet18_cifar10_best.pth'

if SKIP_TRAINING and os.path.exists(CKPT_PATH):
    print(f"⏭️  跳过训练，直接加载已有模型: {CKPT_PATH}")
    checkpoint = torch.load(CKPT_PATH, map_location=device)
    model.load_state_dict(checkpoint['model_state_dict'])
    best_model_state = checkpoint['model_state_dict']
    history = checkpoint.get('history',
                             {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []})
    best_val_acc = checkpoint.get('best_val_acc', 0.0)
    print(f"✓ 模型加载完成，记录的最佳验证准确率: {best_val_acc:.4f}")
else:
    optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
    scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=5, gamma=0.1)

    # 初始化早停（监控验证准确率，越大越好）
    early_stopping = EarlyStopping(patience=PATIENCE, delta=0.001, mode='max', verbose=True)

    history = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}
    best_val_acc = 0.0
    best_model_state = None

    print("开始训练（启用早停机制）...")
    start_time = time.time()

    for epoch in range(1, NUM_EPOCHS + 1):
        print(f"\n{'='*50}\nEpoch {epoch}/{NUM_EPOCHS}\n{'='*50}")
        train_loss, train_acc = train_epoch(model, train_loader, criterion, optimizer, device)
        val_loss, val_acc, _, _ = evaluate(model, val_loader, criterion, device)
        scheduler.step()
        current_lr = optimizer.param_groups[0]['lr']

        history['train_loss'].append(train_loss)
        history['train_acc'].append(train_acc)
        history['val_loss'].append(val_loss)
        history['val_acc'].append(val_acc)

        print(f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f}")
        print(f"Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.4f}")
        print(f"Learning Rate: {current_lr:.6f}")

        # 保存最佳模型
        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_model_state = model.state_dict().copy()
            print(f"✓ 新的最佳模型！验证准确率: {val_acc:.4f}")

        # 早停检查
        early_stopping(val_acc)
        if early_stopping.early_stop:
            print(f"\n⏹️ 早停触发！验证准确率连续 {PATIENCE} 个 epoch 未提升。")
            break

    training_time = time.time() - start_time
    print(f"\n训练完成！总耗时: {training_time/60:.2f} 分钟")
    print(f"最佳验证准确率: {best_val_acc:.4f}")

    # 加载最佳模型用于测试
    model.load_state_dict(best_model_state)

In [ ]:
# ==================== 第六部分：训练曲线绘制 ====================

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].plot(history['train_loss'], label='Train Loss', marker='o')
axes[0].plot(history['val_loss'], label='Val Loss', marker='s')
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Loss')
axes[0].set_title('Loss Curves'); axes[0].legend(); axes[0].grid(alpha=0.3)

axes[1].plot(history['train_acc'], label='Train Acc', marker='o')
axes[1].plot(history['val_acc'], label='Val Acc', marker='s')
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Accuracy')
axes[1].set_title('Accuracy Curves'); axes[1].legend(); axes[1].grid(alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# ==================== 第七部分：测试集详尽评测 ====================

print("\n" + "="*50)
print("测试集评估与指标计算")
print("="*50)

# 在测试集上评估
test_loss, test_acc, test_preds, test_labels = evaluate(model, test_loader, criterion, device)

# 计算加权与宏平均指标
precision_w, recall_w, f1_w, _ = precision_recall_fscore_support(test_labels, test_preds, average='weighted')
precision_m, recall_m, f1_m, _ = precision_recall_fscore_support(test_labels, test_preds, average='macro')

# 计算 Top-3 准确率
def top_k_accuracy(outputs, labels, k=3):
    _, pred_topk = outputs.topk(k, 1, True, True)
    pred_topk = pred_topk.t()
    correct = pred_topk.eq(labels.view(1, -1).expand_as(pred_topk))
    return correct[:k].reshape(-1).float().sum(0, keepdim=True) / labels.size(0)

model.eval()
all_outputs = []
with torch.no_grad():
    for images, _ in test_loader:
        images = images.to(device)
        outputs = model(images)
        all_outputs.append(outputs.cpu())
all_outputs = torch.cat(all_outputs, dim=0)
top3_acc = top_k_accuracy(all_outputs, torch.tensor(test_labels), k=3).item()

print("\n📊 整体指标汇总")
print("-" * 40)
print(f"测试准确率 (Top-1)   : {test_acc:.4f}")
print(f"Top-3 准确率         : {top3_acc:.4f}")
print(f"测试损失             : {test_loss:.4f}")
print(f"加权精确率 (Weighted): {precision_w:.4f}")
print(f"加权召回率 (Weighted): {recall_w:.4f}")
print(f"加权 F1 分数         : {f1_w:.4f}")
print(f"宏平均精确率 (Macro) : {precision_m:.4f}")
print(f"宏平均召回率 (Macro) : {recall_m:.4f}")
print(f"宏平均 F1 分数       : {f1_m:.4f}")

# 各类别详细报告
print("\n📋 各类别分类报告")
print("-" * 40)
print(classification_report(test_labels, test_preds, target_names=CLASS_NAMES, digits=4))

# 混淆矩阵可视化
cm = confusion_matrix(test_labels, test_preds)
plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES)
plt.title('Confusion Matrix - CIFAR-10 Test Set', fontsize=14)
plt.xlabel('Predicted'); plt.ylabel('True')
plt.xticks(rotation=45); plt.yticks(rotation=0)
plt.tight_layout()
plt.show()

In [ ]:
# ==================== 第八部分：关键指标对比柱状图 ====================

metrics_names = ['Top-1 Acc', 'Top-3 Acc', 'Weighted F1', 'Macro F1']
metrics_values = [test_acc, top3_acc, f1_w, f1_m]

plt.figure(figsize=(10, 6))
bars = plt.bar(metrics_names, metrics_values, color=['#2ecc71', '#3498db', '#9b59b6', '#e67e22'])
plt.ylim(0, 1.0)
plt.ylabel('Score')
plt.title('ResNet-18 Performance on CIFAR-10 Test Set')
for bar, val in zip(bars, metrics_values):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
             f'{val:.4f}', ha='center', va='bottom', fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ==================== 第九部分：保存模型权重 ====================

torch.save({
    'model_state_dict': best_model_state,
    'num_classes': NUM_CLASSES,
    'class_names': CLASS_NAMES,
    'history': history,
    'best_val_acc': best_val_acc,
    'test_acc': test_acc,
    'test_f1': f1_w
}, 'resnet18_cifar10_best.pth')

print("模型已保存为 'resnet18_cifar10_best.pth'")

In [ ]:
# ==================== 生成模型下载链接 ====================
import os
from IPython.display import FileLink, display

# 假设已保存模型文件
model_path = 'resnet18_cifar10_best.pth'

if os.path.exists(model_path):
    print(f"✅ 模型文件已就绪，大小: {os.path.getsize(model_path) / 1024**2:.2f} MB")
    display(FileLink(model_path, result_html_prefix="点击此处下载模型: "))
else:
    print("❌ 未找到模型文件，请先保存。")

In [ ]:
# !pip install torchattacks

In [ ]:
# ==================== 优化版本（解决显存不足与速度慢） ====================
# 关键修正：
#   1) 攻击全部在 [0,1] 像素空间进行，归一化放进模型（Normalize 包装层），
#      避免 torchattacks 的 clamp(0,1) 把归一化图像截毁。
#   2) CustomFGSM 与 torchattacks 攻击口径一致，FGSM 同样 clamp 到 [0,1]。
#   3) 修正提前停止条件：按“样本数”而非“batch 数”判断。
import os
import math
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.ticker import PercentFormatter
from tqdm import tqdm
from torch.utils.data import DataLoader, Subset
from torchvision import transforms, datasets
from sklearn.manifold import TSNE
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
import gc

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# CIFAR-10 类别名（攻击单元自包含，避免依赖训练部分的 CLASS_NAMES）
CIFAR10_CLASSES = ['airplane', 'automobile', 'bird', 'cat', 'deer',
                   'dog', 'frog', 'horse', 'ship', 'truck']

# ---------- 全局绘图风格（统一所有攻击图的字体 / 网格 / 留白 / DPI）----------
plt.rcParams.update({
    'figure.dpi': 120, 'savefig.dpi': 300, 'savefig.bbox': 'tight',
    'font.size': 11, 'axes.titlesize': 13, 'axes.titleweight': 'bold',
    'axes.labelsize': 11, 'legend.fontsize': 10,
    'axes.grid': True, 'grid.linestyle': '--', 'grid.alpha': 0.3, 'axes.axisbelow': True,
    'axes.spines.top': False, 'axes.spines.right': False,
    'legend.frameon': True, 'legend.framealpha': 0.9,
})
# 各组（干净 / 各攻击）统一配色（seaborn 'deep' 调色板）
ATTACK_COLORS = {'Clean': '#4C72B0', 'FGSM': '#DD8452',
                 'DeepFool': '#55A868', 'APGD': '#C44E52'}

# ---------- 模型定义 ----------
def create_resnet18(num_classes=10, pretrained=False):
    from torchvision.models import resnet18, ResNet18_Weights
    if pretrained:
        model = resnet18(weights=ResNet18_Weights.IMAGENET1K_V1)
    else:
        model = resnet18(weights=None)
    in_features = model.fc.in_features
    model.fc = nn.Linear(in_features, num_classes)
    return model

model = create_resnet18(num_classes=10, pretrained=False)
CKPT_PATH = '/kaggle/input/notebooks/liangliguo/cifar/resnet18_cifar10_best.pth'
checkpoint = torch.load(CKPT_PATH, map_location=device)
model.load_state_dict(checkpoint['model_state_dict'])
model = model.to(device)
model.eval()
print("Model loaded successfully")

# ---------- 归一化包装层：把归一化放进模型，攻击在 [0,1] 空间进行 ----------
NORM_MEAN = [0.485, 0.456, 0.406]
NORM_STD  = [0.229, 0.224, 0.225]

class Normalize(nn.Module):
    def __init__(self, mean, std):
        super().__init__()
        self.register_buffer('mean', torch.tensor(mean).view(1, 3, 1, 1))
        self.register_buffer('std',  torch.tensor(std).view(1, 3, 1, 1))

    def forward(self, x):
        return (x - self.mean) / self.std

# atk_model 接收 [0,1] 图像，内部先归一化再过 ResNet；
# 训练时网络见到的就是归一化分布，因此干净准确率保持一致。
atk_model = nn.Sequential(Normalize(NORM_MEAN, NORM_STD), model).to(device).eval()
inner_model = model  # 特征 hook 仍挂在内部 ResNet 上

# ---------- 数据加载（不做 Normalize，保持 [0,1]）----------
test_transform = transforms.Compose([
    transforms.Resize(224),
    transforms.ToTensor(),  # → [0,1]，归一化交给 atk_model 内部完成
])
# 本地优先：本地已存在 cifar-10-batches-py 则不联网下载
DATA_ROOT = './data'
_need_download = not os.path.isdir(os.path.join(DATA_ROOT, 'cifar-10-batches-py'))
print(f"{'⬇ 下载数据集到' if _need_download else '✓ 使用本地数据集'}: {os.path.abspath(DATA_ROOT)}")
test_dataset = datasets.CIFAR10(root=DATA_ROOT, train=False,
                                download=_need_download, transform=test_transform)

# 为了加快速度，只使用测试集的一部分（例如前 2000 张）
subset_indices = list(range(2000))
test_subset = Subset(test_dataset, subset_indices)
test_loader = DataLoader(test_subset, batch_size=32, shuffle=False, num_workers=2)  # 减小batch size

# ---------- 防御：高斯平滑（作用于 [0,1] 图像）----------
class GaussianSmoothing(nn.Module):
    def __init__(self, kernel_size=5, sigma=1.0):
        super().__init__()
        self.kernel_size = kernel_size
        self.sigma = sigma
        kernel = self._gaussian_kernel(kernel_size, sigma)
        self.register_buffer('kernel', kernel)

    def _gaussian_kernel(self, size, sigma):
        coords = torch.arange(size, dtype=torch.float32) - size // 2
        g = torch.exp(-(coords**2) / (2 * sigma**2))
        kernel = g[:, None] * g[None, :]
        kernel = kernel / kernel.sum()
        return kernel.view(1, 1, size, size)

    def forward(self, x):
        batch, c, h, w = x.shape
        smoothed = []
        for i in range(c):
            x_ch = x[:, i:i+1, :, :]
            x_ch = torch.nn.functional.conv2d(x_ch, self.kernel, padding=self.kernel_size//2)
            smoothed.append(x_ch)
        return torch.cat(smoothed, dim=1)

# ---------- 自定义 FGSM 攻击（在 [0,1] 空间，结尾 clamp 到 [0,1]）----------
class CustomFGSM:
    def __init__(self, model, eps=0.03):
        self.model = model      # 传入 atk_model（内部含归一化）
        self.eps = eps          # eps 为 [0,1] 像素空间预算

    def __call__(self, images, labels):
        images = images.clone().detach().requires_grad_(True)
        outputs = self.model(images)
        loss = torch.nn.functional.cross_entropy(outputs, labels)
        grad = torch.autograd.grad(loss, images, retain_graph=False, create_graph=False)[0]
        adv_images = images + self.eps * grad.sign()
        adv_images = torch.clamp(adv_images, 0, 1)   # 与 torchattacks 口径一致
        return adv_images.detach()

# ---------- 特征提取（干净样本，前向走 atk_model，hook 挂在内部 ResNet）----------
def extract_features(fwd_model, dataloader, device, max_samples=300):
    fwd_model.eval()
    features = []
    labels = []
    activation = {}
    def hook_fn(module, input, output):
        activation['feat'] = output.detach()
    handle = inner_model.avgpool.register_forward_hook(hook_fn)
    collected = 0
    # 进度条总长按实际要跑的 batch 数计算，凑够 max_samples 即停，进度条到 100%
    total_batches = math.ceil(max_samples / dataloader.batch_size)
    with torch.no_grad():
        for imgs, lbls in tqdm(dataloader, total=total_batches, desc="Extracting features"):
            if collected >= max_samples:
                break
            imgs = imgs.to(device)
            _ = fwd_model(imgs)
            feat = activation['feat'].squeeze(-1).squeeze(-1).cpu()
            features.append(feat)
            labels.append(lbls[:feat.size(0)])
            collected += feat.size(0)
            # 及时释放显存
            del imgs, _
            torch.cuda.empty_cache()
    handle.remove()
    features = torch.cat(features, dim=0)[:max_samples].numpy()
    labels = torch.cat(labels, dim=0)[:max_samples].numpy()
    return features, labels

# ---------- 生成对抗样本（优化版，显存友好） ----------
def generate_adversarial_optimized(attack, name, fwd_model, dataloader, device, max_samples=200):
    """
    优化版：只生成对抗样本和扰动，不同时提取特征（特征单独提取）
    返回：adv_images, orig_images, labels, l2_norms
    """
    fwd_model.eval()
    adv_images_list = []
    orig_images_list = []
    labels_list = []
    l2_norms = []
    collected = 0

    # 进度条总长按实际要跑的 batch 数计算（凑够 max_samples 即停），避免进度条停在低位
    total_batches = math.ceil(max_samples / dataloader.batch_size)
    for imgs, lbls in tqdm(dataloader, total=total_batches, desc=f"Generating {name}"):
        if collected >= max_samples:
            break
        imgs, lbls = imgs.to(device), lbls.to(device)
        # 生成对抗样本（detach：部分 torchattacks 攻击如 APGD 返回的张量仍带 grad）
        adv = attack(imgs, lbls).detach()
        # 计算 L2 扰动
        with torch.no_grad():
            l2 = torch.norm((adv - imgs).view(imgs.size(0), -1), dim=1).cpu()
        adv_images_list.append(adv.cpu())
        orig_images_list.append(imgs.detach().cpu())
        labels_list.append(lbls.cpu())
        l2_norms.append(l2)
        collected += imgs.size(0)
        # 释放显存
        del imgs, lbls, adv
        torch.cuda.empty_cache()
        gc.collect()

    adv_imgs = torch.cat(adv_images_list, dim=0)[:max_samples]
    orig_imgs = torch.cat(orig_images_list, dim=0)[:max_samples]
    labels = torch.cat(labels_list, dim=0)[:max_samples]
    l2_norms = torch.cat(l2_norms, dim=0)[:max_samples].numpy()
    return adv_imgs, orig_imgs, labels, l2_norms

# ---------- 从对抗样本中提取特征 ----------
def extract_features_from_adv(fwd_model, adv_imgs, batch_size=32):
    """分批提取对抗样本的特征，避免显存爆炸"""
    fwd_model.eval()
    features = []
    activation = {}
    def hook_fn(module, input, output):
        activation['feat'] = output.detach()
    handle = inner_model.avgpool.register_forward_hook(hook_fn)

    with torch.no_grad():
        for i in range(0, len(adv_imgs), batch_size):
            batch = adv_imgs[i:i+batch_size].to(device)
            _ = fwd_model(batch)
            feat = activation['feat'].squeeze(-1).squeeze(-1).cpu()
            features.append(feat)
            del batch, _
            torch.cuda.empty_cache()
    handle.remove()
    features = torch.cat(features, dim=0).numpy()
    return features

# ---------- 攻击成功率评估（支持防御） ----------
def evaluate_attack_optimized(attack, name, fwd_model, dataloader, device, num_batches=10, defense=None):
    fwd_model.eval()
    if defense is not None:
        defense = defense.to(device)
    total = 0
    success = 0
    total_l2 = 0.0
    # 进度条总长按实际评估的 batch 数（num_batches）计算
    eval_batches = min(num_batches, len(dataloader))
    for i, (imgs, lbls) in enumerate(tqdm(dataloader, total=eval_batches, desc=f"Evaluating {name}")):
        if i >= num_batches:
            break
        imgs, lbls = imgs.to(device), lbls.to(device)
        adv = attack(imgs, lbls).detach()
        adv_in = defense(adv) if defense is not None else adv
        with torch.no_grad():
            outputs = fwd_model(adv_in)
            pred = outputs.argmax(dim=1)
            success += (pred != lbls).sum().item()
            total += len(lbls)
            l2 = torch.norm((adv - imgs).view(imgs.size(0), -1), dim=1).sum().item()
            total_l2 += l2
        # 释放显存
        del imgs, lbls, adv, outputs
        torch.cuda.empty_cache()
    success_rate = success / total
    avg_l2 = total_l2 / total
    return success_rate, avg_l2

# ---------- 可视化函数 ----------
def tsne_plot(features_dict, title, save_path=None, perplexity=30, seed=42):
    """对多组特征做 标准化 → PCA 预降维 → t-SNE，绘制规范的二维散点图。

    features_dict: {组名: ndarray[N, D]}，如 {"Clean": ..., "FGSM": ..., "APGD": ...}
    """
    names = list(features_dict.keys())
    all_feats = np.concatenate([features_dict[n] for n in names], axis=0)
    group_labels = np.concatenate([[n] * features_dict[n].shape[0] for n in names])

    # 标准化 → PCA 去噪/加速 → t-SNE（现代默认：init='pca', learning_rate='auto'）
    feats_std = StandardScaler().fit_transform(all_feats)
    n_pca = min(50, feats_std.shape[1], feats_std.shape[0] - 1)
    feats_pca = PCA(n_components=n_pca, random_state=seed).fit_transform(feats_std)
    perp = min(perplexity, max(5, (len(feats_pca) - 1) // 3))  # perplexity 需远小于样本数
    tsne = TSNE(n_components=2, random_state=seed, perplexity=perp,
                init='pca', learning_rate='auto')
    emb = tsne.fit_transform(feats_pca)

    # 规范化绘图（配色与其它图统一）
    markers = {'Clean': 'o', 'FGSM': '^', 'DeepFool': 's', 'APGD': 'D'}
    fig, ax = plt.subplots(figsize=(8, 7))
    for idx, name in enumerate(names):
        m = group_labels == name
        ax.scatter(emb[m, 0], emb[m, 1],
                   color=ATTACK_COLORS.get(name), marker=markers.get(name, 'o'),
                   s=30, alpha=0.75, edgecolors='white', linewidths=0.4,
                   label=f"{name} (n={int(m.sum())})")
    ax.set_title(title, pad=12)
    ax.set_xlabel("t-SNE dimension 1")
    ax.set_ylabel("t-SNE dimension 2")
    ax.tick_params(labelsize=9)
    leg = ax.legend(title="Sample type", title_fontsize=11, loc='best')
    leg.get_frame().set_edgecolor('0.8')
    fig.tight_layout()
    if save_path:
        fig.savefig(save_path)
    plt.show()

def show_adv_examples(orig_imgs, adv_imgs, labels, attack_name, num=5,
                      model=None, class_names=None, save_path=None):
    """展示对抗前后的图像、扰动，以及模型在 原图 / 对抗图 上的预测。
    标题用 CIFAR-10 类别名（非编号）；预测正确显示绿色、错误/被攻击成功显示红色。
    """
    if model is None:
        model = atk_model
    if class_names is None:
        class_names = CIFAR10_CLASSES
    model.eval()
    n = min(num, len(orig_imgs))

    # 模型对 原图 / 对抗图 的预测（含置信度）
    with torch.no_grad():
        o_prob = torch.softmax(model(orig_imgs[:n].to(device)), dim=1)
        a_prob = torch.softmax(model(adv_imgs[:n].to(device)), dim=1)
    o_pred, o_conf = o_prob.argmax(1).cpu(), o_prob.max(1).values.cpu()
    a_pred, a_conf = a_prob.argmax(1).cpu(), a_prob.max(1).values.cpu()

    fig, axes = plt.subplots(n, 3, figsize=(10.5, 3.5 * n))
    if n == 1:
        axes = axes.reshape(1, -1)
    for i in range(n):
        true_name = class_names[int(labels[i])]
        o_name = class_names[int(o_pred[i])]
        a_name = class_names[int(a_pred[i])]
        orig = np.clip(orig_imgs[i].detach().cpu().numpy().transpose(1, 2, 0), 0, 1)
        adv = np.clip(adv_imgs[i].detach().cpu().numpy().transpose(1, 2, 0), 0, 1)
        diff = adv - orig
        linf = np.abs(diff).max()
        l2 = float(np.linalg.norm(diff))
        pert = np.abs(diff)
        pert = pert / (pert.max() + 1e-12)

        # 原图：真实类别 + 模型预测（正确绿 / 错误红）
        axes[i, 0].imshow(orig)
        axes[i, 0].set_title(f"Original\nTrue: {true_name}\nPred: {o_name} ({o_conf[i]:.0%})",
                             fontsize=10, color=('green' if o_name == true_name else 'red'))
        axes[i, 0].axis('off')

        # 对抗图：攻击后预测（类别被改=攻击成功 标红，否则标绿）
        attacked = (a_name != true_name)
        axes[i, 1].imshow(adv)
        axes[i, 1].set_title(f"{attack_name} Adv\nPred: {a_name} ({a_conf[i]:.0%})",
                             fontsize=10, color=('red' if attacked else 'green'))
        axes[i, 1].axis('off')

        # 扰动图（绝对值归一化显示，标题给出真实 L2 / L∞ 幅度）
        axes[i, 2].imshow(pert, cmap='inferno')
        axes[i, 2].set_title(f"Perturbation\nL2={l2:.2f}, Linf={linf:.3f}", fontsize=10)
        axes[i, 2].axis('off')

    plt.suptitle(f"{attack_name}: Clean vs Adversarial Predictions",
                 fontsize=13, fontweight='bold')
    fig.tight_layout(rect=[0, 0, 1, 0.98])
    if save_path:
        fig.savefig(save_path, dpi=200)
    plt.show()

def plot_l2_hist(l2_dict, title="Perturbation Magnitude Distribution (L2)", save_path=None):
    """多攻击 L2 扰动分布直方图，带均值竖线，统一配色。
    l2_dict: {attack_name: 1D array of L2 norms}
    """
    fig, ax = plt.subplots(figsize=(8, 5))
    for name, l2 in l2_dict.items():
        c = ATTACK_COLORS.get(name)
        l2 = np.asarray(l2)
        ax.hist(l2, bins=30, alpha=0.55, color=c, edgecolor='white', linewidth=0.4,
                label=f"{name} (mean={l2.mean():.3f})")
        ax.axvline(l2.mean(), color=c, linestyle='--', linewidth=1.3, alpha=0.9)
    ax.set_xlabel("L2 norm of perturbation")
    ax.set_ylabel("Frequency")
    ax.set_title(title)
    ax.legend(title="Attack")
    fig.tight_layout()
    if save_path:
        fig.savefig(save_path)
    plt.show()

def plot_defense_bars(results, title="Defense Effectiveness Comparison", save_path=None):
    """分组柱状图：每种攻击在 原始模型 vs 高斯平滑防御 下的攻击成功率。
    results: {attack_name: (orig_rate, defended_rate)}
    """
    names = list(results.keys())
    orig = [results[n][0] for n in names]
    deff = [results[n][1] for n in names]
    x = np.arange(len(names))
    width = 0.36
    fig, ax = plt.subplots(figsize=(2.2 * len(names) + 2.5, 5))
    b1 = ax.bar(x - width/2, orig, width, label='Original model',
                color='#C44E52', edgecolor='white')
    b2 = ax.bar(x + width/2, deff, width, label='Defended (Gaussian smoothing)',
                color='#4C72B0', edgecolor='white')
    ax.set_xticks(x)
    ax.set_xticklabels(names)
    ax.set_ylabel("Attack success rate")
    ax.set_ylim(0, 1.08)
    ax.yaxis.set_major_formatter(PercentFormatter(xmax=1.0))
    ax.set_title(title)
    ax.grid(axis='x', visible=False)  # 柱状图只保留水平网格
    ax.bar_label(b1, labels=[f"{v:.0%}" for v in orig], padding=3, fontsize=9)
    ax.bar_label(b2, labels=[f"{v:.0%}" for v in deff], padding=3, fontsize=9)
    ax.legend()
    fig.tight_layout()
    if save_path:
        fig.savefig(save_path)
    plt.show()

# ===================== 主流程（优化后） =====================
print("="*60)
print("1. 提取干净样本特征（300个样本）")
clean_feat, clean_label = extract_features(atk_model, test_loader, device, max_samples=300)

print("\n2. 生成 FGSM 对抗样本（200个样本）")
attack_fgsm = CustomFGSM(atk_model, eps=0.03)
fgsm_adv, fgsm_orig, fgsm_lbl, fgsm_l2 = generate_adversarial_optimized(
    attack_fgsm, "FGSM", atk_model, test_loader, device, max_samples=200)

print("\n3. 提取对抗样本特征")
fgsm_feat = extract_features_from_adv(atk_model, fgsm_adv, batch_size=32)

print("\n4. t-SNE 可视化")
features_dict = {"Clean": clean_feat[:200], "FGSM": fgsm_feat}
tsne_plot(features_dict, "t-SNE: Clean vs Adversarial Features", save_path="tsne_clean_vs_adv.png")

print("\n5. 对抗样本可视化")
show_adv_examples(fgsm_orig, fgsm_adv, fgsm_lbl, "FGSM", num=5)

print("\n6. 扰动大小分布")
plot_l2_hist({"FGSM": fgsm_l2}, save_path="l2_distribution.png")

print("\n7. 原始模型攻击成功率评估（使用较小的批次）")
# 为了速度，评估时使用更少的 batch
res_fgsm = evaluate_attack_optimized(attack_fgsm, "FGSM", atk_model, test_loader, device, num_batches=10)
print(f"FGSM      : Success Rate = {res_fgsm[0]:.2%}, Avg L2 = {res_fgsm[1]:.4f}")

print("\n8. 防御后模型评估（高斯平滑）")
defense = GaussianSmoothing(kernel_size=5, sigma=1.0)
res_fgsm_def = evaluate_attack_optimized(attack_fgsm, "FGSM", atk_model, test_loader, device, num_batches=10, defense=defense)
print(f"FGSM (defended)      : Success Rate = {res_fgsm_def[0]:.2%}, Avg L2 = {res_fgsm_def[1]:.4f}")

# 柱状图对比
plot_defense_bars({"FGSM": (res_fgsm[0], res_fgsm_def[0])},
                  save_path="attack_success_comparison.png")

print("\n" + "="*60)
print("总结：")
print("- 攻击在 [0,1] 像素空间进行，归一化放进模型，clamp(0,1) 不再破坏图像。")
print("- 提前停止按样本数判断，真正只跑 200 张，避免整集全跑。")
print("- 高斯平滑防御仍有效降低攻击成功率。")
print("="*60)

In [ ]:
# ==================== 原始（干净）样本的 t-SNE 可视化（按类别上色）====================
# 说明：
#   - 复用上一单元格提取的 clean_feat / clean_label（请先运行 FGSM 单元）。
#   - 与「Clean vs 对抗」的分组 t-SNE 不同，这里只看干净样本，按 10 个真实类别上色，
#     观察 ResNet-18 特征空间中各类别的可分性。
from sklearn.manifold import TSNE
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

def tsne_plot_by_class(features, labels, class_names,
                       title="t-SNE of Clean Samples (by class)",
                       save_path=None, perplexity=30, seed=42):
    """对单组特征做 标准化 → PCA → t-SNE，按真实类别（类名）上色。"""
    labels = np.asarray(labels).astype(int)
    feats_std = StandardScaler().fit_transform(features)
    n_pca = min(50, feats_std.shape[1], feats_std.shape[0] - 1)
    feats_pca = PCA(n_components=n_pca, random_state=seed).fit_transform(feats_std)
    perp = min(perplexity, max(5, (len(feats_pca) - 1) // 3))
    emb = TSNE(n_components=2, random_state=seed, perplexity=perp,
               init='pca', learning_rate='auto').fit_transform(feats_pca)

    cmap = plt.get_cmap('tab10')
    fig, ax = plt.subplots(figsize=(8.5, 7))
    for c in range(len(class_names)):
        m = labels == c
        if not m.any():
            continue
        ax.scatter(emb[m, 0], emb[m, 1], color=cmap(c % 10), s=28, alpha=0.75,
                   edgecolors='white', linewidths=0.4,
                   label=f"{class_names[c]} (n={int(m.sum())})")
    ax.set_title(title, pad=12)
    ax.set_xlabel("t-SNE dimension 1")
    ax.set_ylabel("t-SNE dimension 2")
    ax.tick_params(labelsize=9)
    leg = ax.legend(title="Class", fontsize=9, title_fontsize=11,
                    loc='center left', bbox_to_anchor=(1.01, 0.5))
    leg.get_frame().set_edgecolor('0.8')
    fig.tight_layout()
    if save_path:
        fig.savefig(save_path)
    plt.show()

print("原始样本 t-SNE（按 CIFAR-10 类别上色）")
tsne_plot_by_class(clean_feat, clean_label, CIFAR10_CLASSES,
                   title="t-SNE of Clean CIFAR-10 Samples (by class)",
                   save_path="tsne_clean_by_class.png")

In [ ]:
# ==================== DeepFool 攻击（遵循项目 [0,1] 空间约定）====================
# 说明：
#   - 复用上一单元格定义的 atk_model（内部含归一化）、helper 函数，以及已算出的
#     clean_feat / fgsm_feat / fgsm_l2 / res_fgsm / res_fgsm_def，避免重复计算。
#   - ⚠️ 运行本单元格前，请先运行上一单元格（FGSM 流程），以便复用其变量。
#   - DeepFool 在 [0,1] 像素空间进行，torchattacks 内部 clamp(0,1) 合法。
import torchattacks

print("="*60)
print("1. 生成 DeepFool 对抗样本（steps=20, overshoot=0.02, 200个样本）")
# DeepFool 迭代线性化决策边界，寻找最小 L2 扰动；步数降到 20 兼顾速度与显存。
attack_df = torchattacks.DeepFool(atk_model, steps=20, overshoot=0.02)
df_adv, df_orig, df_lbl, df_l2 = generate_adversarial_optimized(
    attack_df, "DeepFool", atk_model, test_loader, device, max_samples=200)

print("\n2. 提取 DeepFool 对抗样本特征")
df_feat = extract_features_from_adv(atk_model, df_adv, batch_size=32)

print("\n3. t-SNE 可视化（Clean vs FGSM vs DeepFool）")
features_dict = {"Clean": clean_feat[:200], "FGSM": fgsm_feat, "DeepFool": df_feat}
tsne_plot(features_dict, "t-SNE: Clean vs Adversarial Features", save_path="tsne_clean_vs_adv.png")

print("\n4. DeepFool 对抗样本可视化")
show_adv_examples(df_orig, df_adv, df_lbl, "DeepFool", num=5)

print("\n5. 扰动大小分布（FGSM vs DeepFool）")
plot_l2_hist({"FGSM": fgsm_l2, "DeepFool": df_l2}, save_path="l2_distribution.png")

print("\n6. 原始模型攻击成功率评估")
res_df = evaluate_attack_optimized(attack_df, "DeepFool", atk_model, test_loader, device, num_batches=10)
print(f"DeepFool  : Success Rate = {res_df[0]:.2%}, Avg L2 = {res_df[1]:.4f}")

print("\n7. 防御后模型评估（高斯平滑）")
defense = GaussianSmoothing(kernel_size=5, sigma=1.0)
res_df_def = evaluate_attack_optimized(attack_df, "DeepFool", atk_model, test_loader, device, num_batches=10, defense=defense)
print(f"DeepFool (defended)  : Success Rate = {res_df_def[0]:.2%}, Avg L2 = {res_df_def[1]:.4f}")

# 柱状图对比（FGSM vs DeepFool，原始 vs 防御）
plot_defense_bars({"FGSM": (res_fgsm[0], res_fgsm_def[0]),
                   "DeepFool": (res_df[0], res_df_def[0])},
                  save_path="attack_success_comparison.png")

print("\n" + "="*60)
print("DeepFool 小结：")
print("- DeepFool 通过迭代线性化决策边界寻找最小 L2 扰动，通常扰动远小于 FGSM。")
print("- 攻击在 [0,1] 像素空间进行，归一化放进模型，clamp(0,1) 不破坏图像。")
print("- 对比柱状图展示高斯平滑防御对两种攻击的效果差异。")
print("="*60)

In [ ]:
# ==================== APGD 攻击（Auto-PGD，遵循项目 [0,1] 空间约定）====================
# 说明：
#   - 复用前两个单元格定义的 atk_model（内部含归一化）、helper 函数，以及已算出的
#     clean_feat / fgsm_* / res_fgsm* / df_* / res_df* 结果，做三方对比。
#   - ⚠️ 运行本单元格前，请先运行 FGSM 与 DeepFool 两个单元格，以便复用其变量。
#   - APGD 是 Linf 预算攻击（自适应步长 + 动量 + 重启），在 [0,1] 像素空间进行，
#     torchattacks 内部 clamp(0,1) 合法。eps=8/255 为 CIFAR 常用的 Linf 预算。
import torchattacks

EPS_LINF = 8 / 255  # Linf 扰动预算

print("="*60)
print(f"1. 生成 APGD 对抗样本（Linf, eps={EPS_LINF:.4f}, steps=20, 200个样本）")
# APGD 自动调节步长并带重启，是评估鲁棒性的强基线（AutoAttack 的核心组件之一）。
attack_apgd = torchattacks.APGD(atk_model, norm='Linf', eps=EPS_LINF,
                                steps=20, n_restarts=1, loss='ce')
apgd_adv, apgd_orig, apgd_lbl, apgd_l2 = generate_adversarial_optimized(
    attack_apgd, "APGD", atk_model, test_loader, device, max_samples=200)

print("\n2. 提取 APGD 对抗样本特征")
apgd_feat = extract_features_from_adv(atk_model, apgd_adv, batch_size=32)

print("\n3. t-SNE 可视化（Clean vs FGSM vs DeepFool vs APGD）")
features_dict = {"Clean": clean_feat[:200], "FGSM": fgsm_feat, "DeepFool": df_feat, "APGD": apgd_feat}
tsne_plot(features_dict, "t-SNE: Clean vs Adversarial Features", save_path="tsne_clean_vs_adv.png")

print("\n4. APGD 对抗样本可视化")
show_adv_examples(apgd_orig, apgd_adv, apgd_lbl, "APGD", num=5)

print("\n5. 扰动大小分布（FGSM vs DeepFool vs APGD）")
plot_l2_hist({"FGSM": fgsm_l2, "DeepFool": df_l2, "APGD": apgd_l2}, save_path="l2_distribution.png")

print("\n6. 原始模型攻击成功率评估")
res_apgd = evaluate_attack_optimized(attack_apgd, "APGD", atk_model, test_loader, device, num_batches=10)
print(f"APGD  : Success Rate = {res_apgd[0]:.2%}, Avg L2 = {res_apgd[1]:.4f}")

print("\n7. 防御后模型评估（高斯平滑）")
defense = GaussianSmoothing(kernel_size=5, sigma=1.0)
res_apgd_def = evaluate_attack_optimized(attack_apgd, "APGD", atk_model, test_loader, device, num_batches=10, defense=defense)
print(f"APGD (defended)  : Success Rate = {res_apgd_def[0]:.2%}, Avg L2 = {res_apgd_def[1]:.4f}")

# 柱状图对比（FGSM vs DeepFool vs APGD，原始 vs 防御）
plot_defense_bars({"FGSM": (res_fgsm[0], res_fgsm_def[0]),
                   "DeepFool": (res_df[0], res_df_def[0]),
                   "APGD": (res_apgd[0], res_apgd_def[0])},
                  save_path="attack_success_comparison.png")

print("\n" + "="*60)
print("APGD 小结：")
print("- APGD（Auto-PGD）自适应调节步长并带重启，是比 FGSM 更强的 Linf 攻击。")
print("- 相比 DeepFool（最小 L2 扰动），APGD 在固定 Linf 预算下追求更高攻击成功率。")
print("- 攻击在 [0,1] 像素空间进行，归一化放进模型，clamp(0,1) 不破坏图像。")
print("- 对比柱状图展示高斯平滑防御对三种攻击的效果差异。")
print("="*60)